# 007 Hybrid Retrieval Plus Graph

这是 RAG 知识库学习线的第七课。

前面我们已经得到：

```text
chunks.json
triples.json 或 demo triples
ES 索引设计
Neo4j 入图设计
```

本课目标是把召回层串起来：

```text
用户问题
-> BM25 召回
-> 向量召回
-> 图检索
-> 文本证据 + 图证据
-> evidence_package
```

学习目标：

1. 理解 BM25、向量召回、图检索分别解决什么问题。
2. 实现本地 fallback 版关键词召回，保证课程能独立运行。
3. 理解 ES BM25 和 ES kNN 的真实查询形态。
4. 理解 Neo4j 一跳关系查询的真实查询形态。
5. 用 RRF 融合文本召回结果。
6. 生成第八课问答要使用的 `evidence_package.json`。

说明：如果 ES / Neo4j 没有写入数据，本课仍会用本地 chunks/triples 跑通证据包结构。

## 1. 本课的位置

当前阶段：

```text
ES text retrieval
+ ES vector retrieval
+ Neo4j graph retrieval
-> evidence package
```

这一步还不生成最终回答。最终回答放到第八课。

原因是：

```text
召回层负责找证据。
回答层负责基于证据组织语言。
```

两者分开，才容易调试和验证。

## 2. 三种召回的分工

| 召回方式 | 适合什么 | 不适合什么 |
|---|---|---|
| BM25 | 精确词、文件名、专有名词、编号 | 同义表达 |
| 向量召回 | 语义相近问题 | 精确编号、短关键词 |
| 图检索 | 实体关系、跨文档关联 | 全文模糊搜索 |

本课不是用一种方式替代另一种方式，而是组合它们。

## 3. 导入依赖

本课会用到：

```text
openai        -> 生成 query embedding
elasticsearch -> 真实 ES 查询，可选
neo4j         -> 真实图检索，可选
```

In [1]:
import importlib.metadata
import json
import math
import os
import re
from collections import Counter
from hashlib import sha1
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from neo4j import GraphDatabase
from openai import OpenAI

print('openai', importlib.metadata.version('openai'))
print('elasticsearch', importlib.metadata.version('elasticsearch'))
print('neo4j', importlib.metadata.version('neo4j'))

openai 2.36.0
elasticsearch 8.19.3
neo4j 6.2.0


## 4. 加载本地教学数据

本课优先读取：

```text
chunks.json
triples.json
```

如果 `triples.json` 不存在，会构造一个 demo triple。

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
chunks_json_path = generated_dir / 'chunks.json'
triples_json_path = generated_dir / 'triples.json'
evidence_package_path = generated_dir / 'evidence_package.json'

if not chunks_json_path.exists():
    raise FileNotFoundError(f'请先执行第三课生成 chunks.json: {chunks_json_path}')

chunks = json.loads(chunks_json_path.read_text(encoding='utf-8'))
chunks_by_id = {chunk['chunk_id']: chunk for chunk in chunks}

if triples_json_path.exists():
    triple_payload = json.loads(triples_json_path.read_text(encoding='utf-8'))
    triples = triple_payload.get('triples', [])
    print('loaded triples.json')
else:
    first_chunk = chunks[0]
    triples = [
        {
            'subject': '北京市密云水库',
            'predicate': '涉及方案',
            'object': '防御洪水方案',
            'evidence': first_chunk['text'][:120].replace('\n', ' '),
            'confidence': 0.5,
            'doc_id': first_chunk['doc_id'],
            'chunk_id': first_chunk['chunk_id'],
            'page_start': first_chunk['page_start'],
            'page_end': first_chunk['page_end'],
            'file_name': first_chunk['file_name'],
        }
    ]
    print('triples.json not found, using demo triple')

print('doc_id:', doc_id)
print('chunk_count:', len(chunks))
print('triple_count:', len(triples))

loaded triples.json
doc_id: 63b7d4d0675426b5
chunk_count: 143
triple_count: 23


## 5. 配置外部服务

本课默认优先保证教学能跑通。

```python
USE_ES = False
USE_NEO4J = False
```

如果你已经完成第四课写 ES、第六课写 Neo4j，可以改成 `True` 使用真实服务。

In [3]:
RAG_CONFIG = {
    'model_base_url': os.getenv('RAG_MODEL_BASE_URL', 'http://192.168.102.19:8082/v1'),
    #'embedding_model': os.getenv('RAG_EMBEDDING_MODEL', 'Conan-embedding-v1'),
    'embedding_model': 'qwen3-embedding',
    'es_addresses': os.getenv('RAG_ES_ADDRESSES', 'http://192.168.102.19:9200'),
    'es_user': os.getenv('RAG_ES_USER', 'elastic'),
    'es_password': os.getenv('RAG_ES_PASSWORD', 'elastic@2024'),
    'es_index': os.getenv('RAG_ES_INDEX', 'rag_chunks'),
    'neo4j_uri': os.getenv('RAG_NEO4J_URI', 'bolt://192.168.102.19:7687'),
    'neo4j_user': os.getenv('RAG_NEO4J_USER', 'neo4j'),
    'neo4j_password': os.getenv('RAG_NEO4J_PASSWORD', 'neo4j@2025'),
}

USE_ES = True
USE_NEO4J = True

safe_config = dict(RAG_CONFIG)
safe_config['es_password'] = '***'
safe_config['neo4j_password'] = '***'
pprint(safe_config)

{'embedding_model': 'qwen3-embedding',
 'es_addresses': 'http://192.168.102.19:9200',
 'es_index': 'rag_chunks',
 'es_password': '***',
 'es_user': 'elastic',
 'model_base_url': 'http://192.168.102.19:8082/v1',
 'neo4j_password': '***',
 'neo4j_uri': 'bolt://192.168.102.19:7687',
 'neo4j_user': 'neo4j'}


## 6. 用户问题

先用一个和样例 PDF 相关的问题：

```text
密云水库防御洪水方案主要涉及哪些内容？
```

你可以修改这个问题观察召回结果变化。

In [4]:
query = '密引水渠引水流量不能满足泄洪要求应该怎么处理？'
print('query:', query)

query: 密引水渠引水流量不能满足泄洪要求应该怎么处理？


## 7. 本地 fallback：关键词召回

如果 ES 不可用，本地关键词召回能帮助我们先理解流程。

它很简陋，只做：

```text
query 中的关键词在 chunk 中出现越多，分数越高
```

真实项目里应该优先使用 ES BM25。

In [5]:
def tokenize_zh_like(text: str) -> list[str]:
    """教学版中文关键词切分。

    真实生产一般交给 ES analyzer、jieba、HanLP 或领域词典。
    这里为了让离线 notebook 也能看到效果，用 2-4 字滑窗模拟“可能的关键词”。
    """
    raw_tokens = re.findall(r'[\u4e00-\u9fff]+|[a-zA-Z0-9_]+', text)
    tokens = set()
    for raw_token in raw_tokens:
        token = raw_token.lower().strip()
        if not token:
            continue
        if re.fullmatch(r'[\u4e00-\u9fff]+', token):
            if len(token) <= 4:
                tokens.add(token)
            for size in [2, 3, 4]:
                for index in range(0, max(len(token) - size + 1, 0)):
                    tokens.add(token[index:index + size])
        else:
            tokens.add(token)
    return sorted(tokens)


def local_keyword_search(query: str, chunks: list[dict], top_k: int = 8) -> list[dict]:
    query_tokens = tokenize_zh_like(query)
    print('local query tokens:', query_tokens[:30])
    results = []
    for chunk in chunks:
        text = chunk['text']
        score = 0
        for token in query_tokens:
            if token in text.lower():
                score += text.lower().count(token)
        if score > 0:
            results.append({'doc': chunk, 'score': float(score), 'search_type': 'LOCAL_KEYWORD'})
    return sorted(results, key=lambda item: item['score'], reverse=True)[:top_k]

local_keyword_docs = local_keyword_search(query, chunks)

for item in local_keyword_docs[:5]:
    doc = item['doc']
    print(item['score'], doc['chunk_id'], 'page=', doc['page_start'])
    print(doc['text'][:250].replace('\n', ' '))
    print('-' * 80)

local query tokens: ['不能', '不能满', '不能满足', '么处', '么处理', '处理', '密引', '密引水', '密引水渠', '应该', '应该怎', '应该怎么', '引水', '引水流', '引水流量', '引水渠', '引水渠引', '怎么', '怎么处', '怎么处理', '水流', '水流量', '水流量不', '水渠', '水渠引', '水渠引水', '求应', '求应该', '求应该怎', '泄洪']
49.0 63b7d4d0675426b5_chunk_0030 page= 25
6 月1 日至9 月30 日，七孔桥节制闸调度须服从密云水  库水旱灾害防御调度。  （2）调节池调度规程 6 月1 日至9 月30 日，调节池挡水闸闸门全开，调节池 最高水位不超过90.50m，以确保小西库上游农民耕地不被洪  水淹没，小西库上游洪水入调节池后优先引入京密引水渠，  若京密引水渠引水流量不能满足泄洪要求，报北京市水务局  批准后开启调节池泄洪闸，将洪水排入白河河道。  5 调度权限与职责 5.1 调度权限  北京市密云水库管理提出洪水调度方案报送北京市水  务局，抄报北京市水利
--------------------------------------------------------------------------------
41.0 63b7d4d0675426b5_chunk_0017 page= 12
2.2.2 调节池控制水位 6 月1 日至9 月30 日水位不超过90.50m。 3 调度运用计划 6 月1 日至8 月10 日水库限制水位为152.00m，相应库 容30.370 亿m3。8 月11 日至9 月30 日水库限制水位为 154.00m，相应库容33.610 亿m3。其中，8 月11 日至8 月 20 日为过渡期，将库水位控制在152.00m 至154.00m，在过  渡期内逐步抬高。  3.1 6 月1 日至8 月10 日 3.1.1 正常调度规程 （1）当水库水位未达到汛限水位
--------------------------------------------------------------------------------
21.0 63b7d4d0675426b5_c

## 8. 真实 ES BM25 查询形态

如果第四课已经写入 ES，可以打开 `USE_ES = True` 后执行。

BM25 适合精确词：

```text
密云水库
防御洪水
调度方案
```

In [6]:
def create_es_client() -> Elasticsearch:
    return Elasticsearch(
        hosts=RAG_CONFIG['es_addresses'],
        basic_auth=(RAG_CONFIG['es_user'], RAG_CONFIG['es_password']),
        request_timeout=10,
    )


def es_bm25_search(es: Elasticsearch, query: str, size: int = 8) -> list[dict]:
    body = {
        'query': {'match': {'text': {'query': query}}},
        'highlight': {
            'fields': {
                'text': {
                    'pre_tags': ['<strong>'],
                    'post_tags': ['</strong>'],
                    'fragment_size': 160,
                }
            }
        },
    }
    response = es.search(index=RAG_CONFIG['es_index'], body=body, size=size)
    results = []
    for hit in response['hits']['hits']:
        source = hit['_source']
        results.append({'doc': source, 'score': hit['_score'], 'search_type': 'BM25'})
    return results

if USE_ES:
    try:
        es = create_es_client()
        bm25_docs = es_bm25_search(es, query)
        if not bm25_docs:
            print('ES BM25 returned 0 hits, using local keyword fallback')
            bm25_docs = local_keyword_docs
    except Exception as exc:
        print('ES BM25 failed, using local keyword fallback:', repr(exc))
        bm25_docs = local_keyword_docs
else:
    bm25_docs = local_keyword_docs
    print('USE_ES=False, using local keyword fallback as BM25-like docs')

print('bm25_like_count:', len(bm25_docs))

bm25_like_count: 8


/tmp/ipykernel_4013924/240197965.py:22: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  response = es.search(index=RAG_CONFIG['es_index'], body=body, size=size)


## 9. 真实向量召回形态

向量召回流程：

```text
query
-> Conan-embedding-v1
-> ES kNN 查询 vector 字段
```

如果 ES 没有写入向量索引，这一步只能看代码结构，不能得到真实结果。

In [7]:
model_client = OpenAI(api_key=os.getenv('RAG_MODEL_API_KEY', 'EMPTY'), base_url=RAG_CONFIG['model_base_url'])


def embed_query(query: str) -> list[float]:
    response = model_client.embeddings.create(model=RAG_CONFIG['embedding_model'], input=[query])
    return response.data[0].embedding


def es_vector_search(es: Elasticsearch, query: str, k: int = 8, num_candidates: int = 50) -> list[dict]:
    query_vector = embed_query(query)
    body = {
        'knn': {
            'field': 'vector',
            'query_vector': query_vector,
            'k': k,
            'num_candidates': num_candidates,
        },
        'size': k,
    }
    response = es.search(index=RAG_CONFIG['es_index'], body=body)
    results = []
    for hit in response['hits']['hits']:
        source = hit['_source']
        results.append({'doc': source, 'score': hit['_score'], 'search_type': 'VECTOR'})
    return results

if USE_ES:
    vector_docs = es_vector_search(es, query)
else:
    vector_docs = []
    print('USE_ES=False, skip vector search')

print('vector_count:', len(vector_docs))

vector_count: 8


## 10. RRF 融合文本召回结果

BM25 分数和向量分数不是一个尺度，不能直接相加。

RRF 只看各自排名：

```text
rank 越靠前，贡献越高
```

In [8]:
def doc_key(item: dict) -> str:
    doc = item['doc']
    return doc.get('chunk_id') or doc.get('metadata', {}).get('chunk_id') or doc.get('text', '')[:80]


def rrf_fuse(result_lists: list[list[dict]], weights: list[float] | None = None, rank_constant: int = 60) -> list[dict]:
    weights = weights or [1 / len(result_lists)] * len(result_lists)
    scores = {}
    best_docs = {}
    sources = {}

    for list_index, results in enumerate(result_lists):
        weight = weights[list_index] if list_index < len(weights) else 1.0
        for rank, item in enumerate(results, start=1):
            key = doc_key(item)
            best_docs.setdefault(key, item['doc'])
            scores[key] = scores.get(key, 0.0) + weight / (rank_constant + rank)
            sources.setdefault(key, []).append(item.get('search_type', f'list_{list_index}'))

    fused = []
    for key, score in sorted(scores.items(), key=lambda pair: pair[1], reverse=True):
        doc = dict(best_docs[key])
        doc['retrieval_score'] = score
        doc['retrieval_sources'] = sources[key]
        fused.append(doc)
    return fused

text_evidence_docs = rrf_fuse([bm25_docs, vector_docs], weights=[0.4, 0.6]) if vector_docs else rrf_fuse([bm25_docs])

for doc in text_evidence_docs[:5]:
    print(round(doc['retrieval_score'], 6), doc['chunk_id'], doc.get('retrieval_sources'), 'page=', doc.get('page_start'))
    print(doc['text'][:160].replace('\n', ' '))
    print('-' * 80)

0.015806 63b7d4d0675426b5_chunk_0029 ['BM25', 'VECTOR'] page= 24
图7 预报调度2（最大下泄流量3000 m3/s）过程示例  表 7 不同调度措施的调洪成果  预泄流 入库洪 最高水 最大出库流 最高库 超汛限 第10 调度模 量 峰 位出现 削峰率 量 水位 水位时 日水位 式 （m³/s （m³/s 时刻 （%） （m³/s） （m） 间（d） (m) ） ） （h） 规程调 
--------------------------------------------------------------------------------
0.015407 63b7d4d0675426b5_chunk_0017 ['BM25', 'VECTOR'] page= 12
2.2.2 调节池控制水位 6 月1 日至9 月30 日水位不超过90.50m。 3 调度运用计划 6 月1 日至8 月10 日水库限制水位为152.00m，相应库 容30.370 亿m3。8 月11 日至9 月30 日水库限制水位为 154.00m，相应库容33.610 亿m3。其中，8 月11 日至8 月 20 日
--------------------------------------------------------------------------------
0.009677 63b7d4d0675426b5_chunk_0102 ['VECTOR'] page= 81
最大泄量924m3/s。  白河泄水支洞原为白河发电隧洞的施工支洞，后改为泄  万年洪水之永久建筑物，后由于修建第三溢洪道、人防隧洞  等，支洞不再承担泄洪任务。泄水支洞长120.00m，出口闸  门上游段为圆形断面有压隧洞，下游段为城门洞形无压隧  洞。  走马庄隧洞位于走马庄副坝，为千年以上洪水的泄洪任  务而设，
--------------------------------------------------------------------------------
0.009524 63b7d4d0675426b5_chunk_0021 ['VECTOR'] page= 16
（库水位140m），当库水位超过汛限水位152.00m 时，

## 11. 图检索：本地 fallback

如果 Neo4j 还没有写入数据，可以先用 `triples.json` 做本地图检索。

本地规则：

```text
如果 query 中包含 subject/object，或 subject/object 出现在 query 关键词附近，就返回这个 triple。
```

In [9]:
def local_graph_search(query: str, triples: list[dict], top_k: int = 10) -> list[dict]:
    results = []
    lowered_query = query.lower()
    for triple in triples:
        subject = str(triple.get('subject', ''))
        obj = str(triple.get('object', ''))
        predicate = str(triple.get('predicate', ''))
        score = 0
        for value in [subject, obj, predicate]:
            if value and value.lower() in lowered_query:
                score += 2
        for token in tokenize_zh_like(query):
            if token in subject.lower() or token in obj.lower() or token in predicate.lower():
                score += 1
        if score > 0:
            item = dict(triple)
            item['graph_score'] = float(score)
            item['search_type'] = 'LOCAL_GRAPH'
            results.append(item)
    return sorted(results, key=lambda item: item['graph_score'], reverse=True)[:top_k]

local_graph_results = local_graph_search(query, triples)
print('local_graph_count:', len(local_graph_results))
pprint(local_graph_results[:5])

local_graph_count: 10
[{'chunk_id': '63b7d4d0675426b5_chunk_0030',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '京密引水渠引水流量不能满足泄洪要求',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'graph_score': 6.0,
  'object': '京密引水渠',
  'page_end': 25,
  'page_start': 25,
  'predicate': '调度',
  'search_type': 'LOCAL_GRAPH',
  'subject': '调节池'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '3 年一遇洪水控泄流量200m3/s, 最高水位152.93m, 最高水位出现在207h',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'graph_score': 1.0,
  'object': '200m3/s, 最高水位152.93m, 最高水位出现在207h',
  'page_end': 19,
  'page_start': 19,
  'predicate': '结合',
  'search_type': 'LOCAL_GRAPH',
  'subject': '3 年一遇洪水控泄流量'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '3 年一遇洪水控泄流量200m3/s, 预泄水量0.35 亿m3',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'graph_score': 1.0,
  'object': '预泄流量200m3/s, 预泄水量0.35 亿m3',
  'page_end': 19,
  

## 12. 真实 Neo4j 图检索形态

如果第六课已经写入 Neo4j，可以打开 `USE_NEO4J = True` 执行真实一跳查询。

第一版先用实体名直接匹配，后续可以加实体识别和同义词扩展。

In [10]:
def create_neo4j_driver():
    return GraphDatabase.driver(
        RAG_CONFIG['neo4j_uri'],
        auth=(RAG_CONFIG['neo4j_user'], RAG_CONFIG['neo4j_password']),
        connection_timeout=10,
    )


def neo4j_graph_search(driver, entity_names: list[str], limit: int = 20) -> list[dict]:
    cypher = '''
    MATCH (e:Entity)-[r]-(other:Entity)
    WHERE e.name IN $entity_names OR other.name IN $entity_names
    RETURN e.name AS subject,
           r.type AS predicate,
           other.name AS object,
           r.evidence AS evidence,
           r.confidence AS confidence,
           r.evidence_chunk_id AS chunk_id,
           r.page_start AS page_start,
           r.page_end AS page_end
    LIMIT $limit
    '''
    with driver.session() as session:
        return [dict(record) for record in session.run(cypher, entity_names=entity_names, limit=limit)]

query_entity_candidates = []
for triple in triples:
    for name in [triple.get('subject'), triple.get('object')]:
        if name and name in query:
            query_entity_candidates.append(name)
query_entity_candidates = sorted(set(query_entity_candidates))

if USE_NEO4J and query_entity_candidates:
    neo4j_driver = create_neo4j_driver()
    graph_results = neo4j_graph_search(neo4j_driver, query_entity_candidates)
    neo4j_driver.close()
else:
    graph_results = local_graph_results
    print('using local graph fallback')

print('query_entity_candidates:', query_entity_candidates)
print('graph_result_count:', len(graph_results))
pprint(graph_results[:5])

using local graph fallback
query_entity_candidates: []
graph_result_count: 10
[{'chunk_id': '63b7d4d0675426b5_chunk_0030',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '京密引水渠引水流量不能满足泄洪要求',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'graph_score': 6.0,
  'object': '京密引水渠',
  'page_end': 25,
  'page_start': 25,
  'predicate': '调度',
  'search_type': 'LOCAL_GRAPH',
  'subject': '调节池'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '3 年一遇洪水控泄流量200m3/s, 最高水位152.93m, 最高水位出现在207h',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'graph_score': 1.0,
  'object': '200m3/s, 最高水位152.93m, 最高水位出现在207h',
  'page_end': 19,
  'page_start': 19,
  'predicate': '结合',
  'search_type': 'LOCAL_GRAPH',
  'subject': '3 年一遇洪水控泄流量'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0024',
  'confidence': 0.9,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '3 年一遇洪水控泄流量200m3/s, 预泄水量0.35 亿m3',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'graph_score': 1.0,
  'ob

## 13. 构造 evidence_package

第八课问答生成不应该直接面对散乱的召回结果。

我们要把证据组织成统一结构：

```text
question
text_evidence
graph_evidence
```

In [11]:
def build_text_evidence(docs: list[dict], limit: int = 6) -> list[dict]:
    evidence = []
    for doc in docs[:limit]:
        evidence.append(
            {
                'chunk_id': doc['chunk_id'],
                'doc_id': doc['doc_id'],
                'file_name': doc['file_name'],
                'page_start': doc.get('page_start'),
                'page_end': doc.get('page_end'),
                'text': doc['text'],
                'retrieval_score': doc.get('retrieval_score'),
                'retrieval_sources': doc.get('retrieval_sources', []),
            }
        )
    return evidence


def build_graph_evidence(results: list[dict], limit: int = 10) -> list[dict]:
    evidence = []
    for item in results[:limit]:
        evidence.append(
            {
                'subject': item.get('subject'),
                'predicate': item.get('predicate'),
                'object': item.get('object'),
                'evidence': item.get('evidence'),
                'confidence': item.get('confidence'),
                'chunk_id': item.get('chunk_id'),
                'page_start': item.get('page_start'),
                'page_end': item.get('page_end'),
                'search_type': item.get('search_type', 'GRAPH'),
            }
        )
    return evidence

evidence_package = {
    'question': query,
    'doc_id': doc_id,
    'text_evidence': build_text_evidence(text_evidence_docs),
    'graph_evidence': build_graph_evidence(graph_results),
}

pprint(evidence_package)

{'doc_id': '63b7d4d0675426b5',
 'graph_evidence': [{'chunk_id': '63b7d4d0675426b5_chunk_0030',
                     'confidence': 0.9,
                     'evidence': '京密引水渠引水流量不能满足泄洪要求',
                     'object': '京密引水渠',
                     'page_end': 25,
                     'page_start': 25,
                     'predicate': '调度',
                     'search_type': 'LOCAL_GRAPH',
                     'subject': '调节池'},
                    {'chunk_id': '63b7d4d0675426b5_chunk_0024',
                     'confidence': 0.9,
                     'evidence': '3 年一遇洪水控泄流量200m3/s, 最高水位152.93m, 最高水位出现在207h',
                     'object': '200m3/s, 最高水位152.93m, 最高水位出现在207h',
                     'page_end': 19,
                     'page_start': 19,
                     'predicate': '结合',
                     'search_type': 'LOCAL_GRAPH',
                     'subject': '3 年一遇洪水控泄流量'},
                    {'chunk_id': '63b7d4d0675426b5_chunk_0024',
                     'confidence

## 14. 保存 evidence_package.json

第八课会读取这个文件，基于证据包生成回答。

In [12]:
evidence_package_path.write_text(json.dumps(evidence_package, ensure_ascii=False, indent=2), encoding='utf-8')

print('evidence_package_path:', evidence_package_path)
print('size KB:', round(evidence_package_path.stat().st_size / 1024, 2))

evidence_package_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/evidence_package.json
size KB: 13.99


## 15. 本课小结

本课完成：

```text
query
-> BM25-like retrieval
-> vector retrieval shape
-> graph retrieval shape
-> RRF fusion
-> evidence_package.json
```

关键结论：

```text
多路召回的目标不是让每路都给最终答案。
它们的目标是提供互补证据，然后统一交给回答层。
```

下一课会进入：

```text
evidence_package.json
-> 基于证据生成回答
-> 引用 chunk_id / page
-> 检查回答是否有证据支持
```

## 16. 练习

请你回答：

1. BM25 和向量召回为什么不能直接比较原始分数？
2. 图检索结果为什么不直接混进 RRF 排名？
3. `evidence_package` 为什么要同时保留 text evidence 和 graph evidence？
4. 如果 `graph_evidence` 为空，是否代表不能回答？为什么？